# Omnichannel ROPO Analytics Pipeline

## Phase 4: Analytical Warehouse Tier

### Technology Stack
- Python
- Pandas
- PostgreSQL
- SQLAlchemy
- psycopg2

# Import Libraries

In [1]:
import pandas as pd
import numpy as np

# Load Dataset 

In [3]:
df = pd.read_csv("../data/retailhive_10k.csv")

# View first 5 rows

In [4]:
df.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102


Create Dim_Product

In [9]:
# ==========================
# STEP 1 : Create Product Dimension
# ==========================

dim_product = (
    df[["product_title", "rating"]]
    .drop_duplicates(subset=["product_title"])
    .reset_index(drop=True)
)

# Create Product ID
dim_product.insert(0, "product_id", range(1, len(dim_product) + 1))

In [10]:
# Verify 

dim_product.head()

,product_id,product_title,rating
0,1,The Dirty Little Secrets of Getting Your Dream...,Four
1,2,Mesaerion: The Best Science Fiction Stories 18...,One
2,3,Libertarianism for Beginners,Two
3,4,On a Midnight Clear,Three
4,5,Online Marketing for Busy Authors: A Step-By-S...,One


In [16]:
dim_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     100 non-null    int64 
 1   product_title  100 non-null    object
 2   rating         100 non-null    object
dtypes: int64(1), object(2)
memory usage: 2.5+ KB


In [12]:
len(dim_product)

100

Create Dim_Customer

In [13]:
# ==========================
# STEP 2 : Create Customer Dimension
# ==========================

dim_customer = (
    df[["customer_id"]]
    .drop_duplicates()
    .sort_values("customer_id")
    .reset_index(drop=True)
)

In [15]:
#Verify

dim_customer.head()

,customer_id
0,10000
1,10005
2,10011
3,10016
4,10019


In [17]:
dim_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10358 entries, 0 to 10357
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  10358 non-null  int64
dtypes: int64(1)
memory usage: 81.1 KB


In [18]:
len(dim_customer)

10358

Create Dim_Channel

In [19]:
# ==========================
# STEP 3 : Create Channel Dimension
# ==========================

dim_channel = (
    df[["channel"]]
    .drop_duplicates()
    .sort_values("channel")
    .reset_index(drop=True)
)

# Create Channel ID
dim_channel.insert(0, "channel_id", range(1, len(dim_channel) + 1))

In [20]:
#verify

dim_channel.head()

,channel_id,channel
0,1,Mobile
1,2,Online
2,3,Store


In [21]:
dim_channel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   channel_id  3 non-null      int64 
 1   channel     3 non-null      object
dtypes: int64(1), object(1)
memory usage: 180.0+ bytes


In [22]:
len(dim_channel)

3

Create Dim_Region

In [23]:
# ==========================
# STEP 4 : Create Region Dimension
# ==========================

dim_region = (
    df[["region"]]
    .drop_duplicates()
    .sort_values("region")
    .reset_index(drop=True)
)

# Create Region ID
dim_region.insert(0, "region_id", range(1, len(dim_region) + 1))

In [25]:
#verify

dim_region.head()

,region_id,region
0,1,Asia
1,2,Europe
2,3,US


In [26]:
dim_region.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   region_id  3 non-null      int64 
 1   region     3 non-null      object
dtypes: int64(1), object(1)
memory usage: 180.0+ bytes


In [27]:
len(dim_region)

3

Create Dim_Date

In [29]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

In [30]:
# Verify
df["date"].dtype

dtype('<M8[ns]')

In [31]:
# ==========================
# STEP 5 : Create Date Dimension
# ==========================

dim_date = (
    df[["date"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

# Create Date ID
dim_date.insert(0, "date_id", range(1, len(dim_date) + 1))

# Create additional date attributes
dim_date["day"] = dim_date["date"].dt.day
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["year"] = dim_date["date"].dt.year
dim_date["weekday"] = dim_date["date"].dt.day_name()

In [32]:
# Verify

dim_date.head()

,date_id,date,day,month,month_name,quarter,year,weekday
0,1,2019-01-01,1,1,January,1,2019,Tuesday
1,2,2019-01-02,2,1,January,1,2019,Wednesday
2,3,2019-01-03,3,1,January,1,2019,Thursday
3,4,2019-01-04,4,1,January,1,2019,Friday
4,5,2019-01-05,5,1,January,1,2019,Saturday


In [33]:
dim_date.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2527 entries, 0 to 2526
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date_id     2527 non-null   int64         
 1   date        2527 non-null   datetime64[ns]
 2   day         2527 non-null   int64         
 3   month       2527 non-null   int64         
 4   month_name  2527 non-null   object        
 5   quarter     2527 non-null   int64         
 6   year        2527 non-null   int64         
 7   weekday     2527 non-null   object        
dtypes: datetime64[ns](1), int64(5), object(2)
memory usage: 158.1+ KB


In [34]:
len(dim_date)

2527

Create Fact_ROPO

In [35]:
# Step 1: Create a working copy

fact_ropo = df.copy()

In [36]:
# Verify
fact_ropo.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102


In [37]:
# Add product ID

fact_ropo = fact_ropo.merge(
    dim_product,
    on=["product_title", "rating"],
    how="left"
)

In [38]:
# Verify

fact_ropo.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue,product_id
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202,1
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634,2
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464,3
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351,4
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102,5


In [39]:
# Step 3: Add Channel ID

fact_ropo = fact_ropo.merge(
    dim_channel,
    on="channel",
    how="left"
)

In [41]:
# Verify
fact_ropo.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue,product_id,channel_id
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202,1,3
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634,2,2
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464,3,2
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351,4,1
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102,5,1


In [42]:
# Step 4: Add Region ID

fact_ropo = fact_ropo.merge(
    dim_region,
    on="region",
    how="left"
)

In [43]:
# Verify

fact_ropo.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue,product_id,channel_id,region_id
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202,1,3,3
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634,2,2,1
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464,3,2,3
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351,4,1,3
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102,5,1,2


In [44]:
# Step 5: Add Date ID

fact_ropo = fact_ropo.merge(
    dim_date[["date_id", "date"]],
    on="date",
    how="left"
)

In [45]:
# Verify 

fact_ropo.head()

,product_title,rating,transaction_id,date,channel,region,customer_id,quantity,price,revenue,product_id,channel_id,region_id,date_id
0,The Dirty Little Secrets of Getting Your Dream...,Four,1,2020-05-02,Store,US,13068,3,35.370401,106.111202,1,3,3,480
1,Mesaerion: The Best Science Fiction Stories 18...,One,2,2025-07-20,Online,Asia,40027,4,32.506908,130.027634,2,2,1,2365
2,Libertarianism for Beginners,Two,3,2024-07-30,Online,US,40294,3,47.085821,141.257464,3,2,3,2016
3,On a Midnight Clear,Three,4,2020-04-08,Mobile,US,19924,3,14.811784,44.435351,4,1,3,456
4,Online Marketing for Busy Authors: A Step-By-S...,One,5,2024-09-27,Mobile,Europe,40383,3,51.179701,153.539102,5,1,2,2075


In [46]:
# Step 6: Remove Dimension Attributes

fact_ropo = fact_ropo.drop(
    columns=[
        "product_title",
        "rating",
        "channel",
        "region",
        "date"
    ]
)

In [47]:
# Verify

fact_ropo.head()

,transaction_id,customer_id,quantity,price,revenue,product_id,channel_id,region_id,date_id
0,1,13068,3,35.370401,106.111202,1,3,3,480
1,2,40027,4,32.506908,130.027634,2,2,1,2365
2,3,40294,3,47.085821,141.257464,3,2,3,2016
3,4,19924,3,14.811784,44.435351,4,1,3,456
4,5,40383,3,51.179701,153.539102,5,1,2,2075


In [48]:
# Step 7: Record the fact table

fact_ropo = fact_ropo[
    [
        "transaction_id",
        "customer_id",
        "product_id",
        "date_id",
        "channel_id",
        "region_id",
        "quantity",
        "price",
        "revenue"
    ]
]

In [49]:
# Verify

fact_ropo.head()

,transaction_id,customer_id,product_id,date_id,channel_id,region_id,quantity,price,revenue
0,1,13068,1,480,3,3,3,35.370401,106.111202
1,2,40027,2,2365,2,1,4,32.506908,130.027634
2,3,40294,3,2016,2,3,3,47.085821,141.257464
3,4,19924,4,456,1,3,3,14.811784,44.435351
4,5,40383,5,2075,1,2,3,51.179701,153.539102


In [50]:
fact_ropo.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  12000 non-null  int64  
 1   customer_id     12000 non-null  int64  
 2   product_id      12000 non-null  int64  
 3   date_id         12000 non-null  int64  
 4   channel_id      12000 non-null  int64  
 5   region_id       12000 non-null  int64  
 6   quantity        12000 non-null  int64  
 7   price           12000 non-null  float64
 8   revenue         12000 non-null  float64
dtypes: float64(2), int64(7)
memory usage: 937.5 KB


In [51]:
len(fact_ropo)

12000

# Import Libraries

In [52]:
from sqlalchemy import create_engine
import psycopg2

Create the Connection

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="your_password",   # replace with your own PostgreSQL password
    host="localhost",
    port=5432,
    database="ROPO_Analytics"
)

engine = create_engine(url)

print("Connected Successfully!")

Connected Successfully!


Load the Dimension Tables into PostgreSQL

In [58]:
dim_product.to_sql(
    "dim_product",
    con=engine,
    if_exists="replace",
    index=False
)

print("Dim_Product loaded successfully!")

Dim_Product loaded successfully!


In [59]:
dim_customer.to_sql(
    "dim_customer",
    con=engine,
    if_exists="replace",
    index=False
)

print("Dim_Customer Loaded")

Dim_Customer Loaded


In [60]:
dim_channel.to_sql(
    "dim_channel",
    con=engine,
    if_exists="replace",
    index=False
)

print("Dim_Channel Loaded")

Dim_Channel Loaded


In [61]:
dim_region.to_sql(
    "dim_region",
    con=engine,
    if_exists="replace",
    index=False
)

print("Dim_Region Loaded")

Dim_Region Loaded


In [62]:
dim_date.to_sql(
    "dim_date",
    con=engine,
    if_exists="replace",
    index=False
)

print("Dim_Date Loaded")

Dim_Date Loaded


In [63]:
fact_ropo.to_sql(
    "fact_ropo",
    con=engine,
    if_exists="replace",
    index=False
)

print("Fact_ROPO Loaded")

Fact_ROPO Loaded
